# Outlier removal with SOR

This notebook shows the basic usage of Statistical Outlier Removal (SOR) for filtering point clouds.

**Implemented by**  
Ronald Tabernig ([@tabernig](https://github.com/tabernig))

**Author(s) of the method**
- SOR: Rusu et al. (2008)

**Original publication of the method**
- Rusu, R.B., Marton, Z.C., Blodow, N., Dolha, M., Beetz, M. (2008). *Towards 3D point cloud based object maps for household environments*. Robotics and Autonomous Systems, 56, 927-941. https://doi.org/10.1016/j.robot.2008.08.005

## **Method description**

This workflow applies SOR to detect points with unusually large local neighbor distances.

Pipeline:
1. Run SOR and keep the inlier/outlier flag.
2. Export the inlier/outlier flag to LAS for inspection.

In [ ]:
import py4dgeo
import numpy as np

from py4dgeo.util import find_file

py4dgeo.enable_trace(False)
py4dgeo.enable_timeit(False)

In [ ]:
# Fetch test data
test_filename = "trier_sim.zip"
original_file = find_file(test_filename)
print(f"File downloaded to: {original_file}")

In [ ]:
before_rockfall_file = "trier_sim_epoch_0.laz"

## Statistical Outlier Removal (SOR)

We define input/output paths and run SOR on our epoch with `k` and `std_dev_multiplier`.

`k` controls how many nearest neighbors are used for each point. `std_dev_multiplier` controls how strict the outlier classification is.

`remove_points=False` keeps all points and returns a per-point inlier/outlier flag. This flag is written to `SOR_out`.


In [ ]:
SOR_out = "outlier_removal_SOR.laz"

dims = {"return_number": "return_number", "number_of_returns": "number_of_returns"}

k = 8
std_dev_multiplier = 1.0
# Let's not remove the points directly but store the inlier/outlier flag instead.
remove_points = False

search_points_epoch = py4dgeo.read_from_las(
    find_file(before_rockfall_file), additional_dimensions=dims
)
search_points_epoch, inlier_outlier = py4dgeo.statistical_outlier_removal(
    search_points_epoch,
    k=k,
    std_dev_multiplier=std_dev_multiplier,
    remove_points=remove_points,
)

print(f"Outliers: {inlier_outlier.sum()} / {len(inlier_outlier)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap

plot_every_nth_point = 10
view_elev = 20
view_azim = -65
sample_cloud = search_points_epoch.cloud[::plot_every_nth_point]
sample_flags = inlier_outlier[::plot_every_nth_point]

fig = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig.add_subplot(1, 1, 1, projection="3d")

outlier_cmap = ListedColormap(["#DADADA36", "#FF0000"])
outlier_norm = BoundaryNorm([-0.5, 0.5, 1.5], outlier_cmap.N)
outlier_plot = ax.scatter(
    sample_cloud[:, 0],
    sample_cloud[:, 1],
    sample_cloud[:, 2],
    c=sample_flags,
    cmap=outlier_cmap,
    norm=outlier_norm,
    s=1,
)
ax.set_title("SOR classification")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.view_init(elev=view_elev, azim=view_azim)
ax.set_box_aspect(np.ptp(sample_cloud, axis=0))
colorbar = fig.colorbar(outlier_plot, ax=ax, ticks=[0, 1])
colorbar.ax.set_yticklabels(["Inlier", "Outlier"])

plt.show()

## Exporting SOR classification to LAS for further analysis

We store the SOR inlier/outlier flag as a LAS attribute via `py4dgeo.Vapc`.
The resulting `SOR_out` file can be inspected in CloudCompare (or similar tools) for further visual inspection.


In [ ]:
vapc = py4dgeo.Vapc(
    search_points_epoch, voxel_size=0.01
)  # Voxel size is not used here.
vapc.out[f"SOR_outlier_k_{k}"] = inlier_outlier
vapc.save_as_las(SOR_out)

## References

- Rusu, R.B., Marton, Z.C., Blodow, N., Dolha, M., Beetz, M. (2008). *Towards 3D point cloud based object maps for household environments*. Robotics and Autonomous Systems, 56, 927-941. https://doi.org/10.1016/j.robot.2008.08.005
